In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import warnings


In [ ]:
# Configuration
np.random.seed(42)  # Reproducibility
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Population and time parameters
N_PHILIPPINES = 108_000_000  # 2020 census
START_DATE = '2020-03-01'
END_DATE = '2021-12-31'
DT = 1.0  # day

print(f"Analysis: Philippine SIR Model ({START_DATE} to {END_DATE})")
print(f"Population: {N_PHILIPPINES:,}")
print(f"Step size: {DT} day")

In [ ]:
def load_covid_data():
    """
    Load Philippine COVID-19 data from JHU CSSE repository.

    Returns:
        dates: array of datetime objects
        cases: array of cumulative confirmed cases
    """
    # JHU CSSE time series URL
    url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
           "master/csse_covid_19_data/csse_covid_19_time_series/"
           "time_series_covid19_confirmed_global.csv")

    # url= ("/content/time_series_covid19_confirmed_global.csv")

    # Load data
    df = pd.read_csv(url)

    # Extract Philippines data
    philippines = df[df['Country/Region'] == 'Philippines'].iloc[0]

    # Drop non-date columns
    date_cols = [c for c in df.columns if c not in
                 ['Province/State', 'Country/Region', 'Lat', 'Long']]

    # Parse dates and values
    dates = pd.to_datetime(date_cols)
    cases = philippines[date_cols].values.astype(float)

    # Filter to study period
    start_dt = pd.to_datetime(START_DATE)
    end_dt = pd.to_datetime(END_DATE)
    mask = (dates >= start_dt) & (dates <= end_dt)

    dates = dates[mask]
    cases = cases[mask]

    print(f"Loaded {len(cases)} daily observations")
    print(f"Cases range: {cases[0]:.0f} to {cases[-1]:,.0f}")

    return dates.values, cases

In [ ]:
def prepare_initial_conditions(cases_0):
    """
    Set up SIR initial conditions from first observation.

    Args:
        cases_0: cumulative cases at t=0

    Returns:
        S0, I0, R0: initial compartment values
    """
    I0 = cases_0  # Interpret as cumulative incidence
    R0 = 0.0      # No initial recovery
    S0 = N_PHILIPPINES - I0 - R0

    return S0, I0, R0

In [ ]:
def sir_derivatives(S, I, R, beta, gamma, N):
    """
    Compute SIR model derivatives.

    Args:
        S, I, R: current compartment values
        beta: transmission rate (1/day)
        gamma: recovery rate (1/day)
        N: total population

    Returns:
        dS, dI, dR: time derivatives
    """
    # Force of infection
    lambda_ = beta * I / N

    # Derivatives
    dS = -lambda_ * S
    dI = lambda_ * S - gamma * I
    dR = gamma * I

    return dS, dI, dR

In [ ]:
def euler_sir(S0, I0, R0, beta, gamma, N, dt, n_steps):
    """
    Explicit Euler integration of SIR model.

    Args:
        S0, I0, R0: initial conditions
        beta, gamma: model parameters
        N: total population
        dt: time step
        n_steps: number of steps to integrate

    Returns:
        S, I, R: arrays of compartment values (length n_steps+1)
    """
    # Initialize arrays
    S = np.zeros(n_steps + 1)
    I = np.zeros(n_steps + 1)
    R = np.zeros(n_steps + 1)

    S[0], I[0], R[0] = S0, I0, R0

    # Time stepping
    for n in range(n_steps):
        # Compute derivatives at current state
        dS, dI, dR = sir_derivatives(S[n], I[n], R[n], beta, gamma, N)

        # Explicit update
        S[n+1] = S[n] + dt * dS
        I[n+1] = I[n] + dt * dI
        # Enforce conservation: R = N - S - I
        R[n+1] = N - S[n+1] - I[n+1]

    return S, I, R

In [ ]:
def rk4_sir(S0, I0, R0, beta, gamma, N, dt, n_steps):
    """
    Classical RK4 integration of SIR model.

    Args: (same as euler_sir)

    Returns: (same as euler_sir)
    """
    # Initialize arrays
    S = np.zeros(n_steps + 1)
    I = np.zeros(n_steps + 1)
    R = np.zeros(n_steps + 1)

    S[0], I[0], R[0] = S0, I0, R0

    # Helper for derivatives at given state
    def derivs(s, i, r):
        return sir_derivatives(s, i, r, beta, gamma, N)

    # Time stepping
    for n in range(n_steps):
        # Stage 1: initial point
        k1_S, k1_I, k1_R = derivs(S[n], I[n], R[n])

        # Stage 2: midpoint with k1
        k2_S, k2_I, k2_R = derivs(
            S[n] + 0.5*dt*k1_S,
            I[n] + 0.5*dt*k1_I,
            R[n] + 0.5*dt*k1_R
        )

        # Stage 3: midpoint with k2
        k3_S, k3_I, k3_R = derivs(
            S[n] + 0.5*dt*k2_S,
            I[n] + 0.5*dt*k2_I,
            R[n] + 0.5*dt*k2_R
        )

        # Stage 4: endpoint with k3
        k4_S, k4_I, k4_R = derivs(
            S[n] + dt*k3_S,
            I[n] + dt*k3_I,
            R[n] + dt*k3_R
        )

        # Weighted combination
        S[n+1] = S[n] + (dt/6) * (k1_S + 2*k2_S + 2*k3_S + k4_S)
        I[n+1] = I[n] + (dt/6) * (k1_I + 2*k2_I + 2*k3_I + k4_I)
        # Enforce conservation
        R[n+1] = N - S[n+1] - I[n+1]

    return S, I, R

In [ ]:
def create_objective(cases_observed, S0, I0, R0, N, dt, n_steps, method='rk4'):
    """
    Create least-squares objective function for parameter estimation.

    Args:
        cases_observed: array of observed cumulative cases
        S0, I0, R0: initial conditions
        N: total population
        dt: time step
        n_steps: number of integration steps
        method: 'euler' or 'rk4'

    Returns:
        objective function for optimization
    """
    solver = rk4_sir if method == 'rk4' else euler_sir

    def objective(params):
        """
        Compute sum of squared errors.

        Args:
            params: [log_beta, log_gamma] (log-transformed for positivity)

        Returns:
            sum of squared errors
        """
        # Transform to natural scale
        beta = np.exp(params[0])
        gamma = np.exp(params[1])

        # Integrate SIR model
        S, I, R = solver(S0, I0, R0, beta, gamma, N, dt, n_steps)

        # Compute squared error (I vs observed cases)
        # Note: I represents cumulative incidence in this interpretation
        error = I - cases_observed
        sse = np.sum(error**2)

        return sse

    return objective

In [ ]:
def estimate_parameters(cases_observed, S0, I0, R0, N, dt, n_steps,
                        method='rk4', n_restarts=5):
    """
    Estimate SIR parameters using Nelder-Mead optimization with restarts.

    Args: (as above)

    Returns:
        beta_hat, gamma_hat: estimated parameters
        R0_hat: implied basic reproduction number
        optimization_result: full scipy result object
    """
    # Create objective
    obj = create_objective(cases_observed, S0, I0, R0, N, dt, n_steps, method)

    # Multiple random restarts
    best_result = None
    best_value = np.inf

    for i in range(n_restarts):
        # Random initialization in plausible range
        if i == 0:
            # Start near literature values
            x0 = np.log([0.3, 0.1])
        else:
            x0 = np.log(np.random.uniform([0.05, 0.03], [1.0, 0.3]))

        # Optimize
        result = minimize(
            obj,
            x0,
            method='Nelder-Mead',
            options={'maxiter': 1000, 'xatol': 1e-6, 'fatol': 1e-6}
        )

        if result.fun < best_value:
            best_value = result.fun
            best_result = result

    # Extract estimates
    beta_hat = np.exp(best_result.x[0])
    gamma_hat = np.exp(best_result.x[1])
    R0_hat = beta_hat / gamma_hat

    print(f"\nParameter estimates ({method}):")
    print(f"  β̂ = {beta_hat:.4f} day⁻¹")
    print(f"  γ̂ = {gamma_hat:.4f} day⁻¹")
    print(f"  R₀ = {R0_hat:.3f}")
    print(f"  SSE = {best_result.fun:.2e}")
    print(f"  Iterations: {best_result.nit}")

    return beta_hat, gamma_hat, R0_hat, best_result

In [ ]:
def compute_error_metrics(I_model, I_observed):
    """
    Compute comprehensive error metrics.

    Args:
        I_model: model-predicted infections
        I_observed: observed infections

    Returns:
        dict of error metrics
    """
    # Pointwise errors
    error = I_model - I_observed
    abs_error = np.abs(error)

    # Aggregate metrics
    mae = np.mean(abs_error)                    # Mean Absolute Error
    rmse = np.sqrt(np.mean(error**2))           # Root Mean Square Error
    max_error = np.max(abs_error)               # Maximum Absolute Error
    mape = np.mean(abs_error / (I_observed + 1)) * 100  # Mean Abs % Error

    # Distribution statistics
    error_std = np.std(error)
    error_skew = np.mean((error - np.mean(error))**3) / error_std**3

    metrics = {
        'MAE': mae,
        'RMSE': rmse,
        'Max_Error': max_error,
        'MAPE': mape,
        'Error_Std': error_std,
        'Error_Skew': error_skew,
        'Error_Array': error,
        'Abs_Error_Array': abs_error
    }

    return metrics

In [ ]:
def compare_methods(I_euler, I_rk4, I_observed):
    """
    Direct comparison of Euler and RK4 predictions.

    Args:
        I_euler, I_rk4: model predictions from each method
        I_observed: observed data

    Returns:
        comparison statistics
    """
    # Method-to-method divergence
    method_diff = np.abs(I_euler - I_rk4)

    comparison = {
        'Max_Method_Diff': np.max(method_diff),
        'Mean_Method_Diff': np.mean(method_diff),
        'Method_Correlation': np.corrcoef(I_euler, I_rk4)[0,1],
        'Euler_vs_Data': compute_error_metrics(I_euler, I_observed),
        'RK4_vs_Data': compute_error_metrics(I_rk4, I_observed)
    }

    return comparison

In [ ]:
# ============================================================
# STEP-SIZE SENSITIVITY ANALYSIS
# ============================================================

def step_size_sensitivity(S0, I0, R0, beta, gamma, N, n_days, step_sizes):
    """
    Analyze how different step sizes (dt) affect Euler and RK4 solutions.

    This answers: "Does the model blow up or change a lot when we use
    bigger or smaller time steps?"

    Args:
        S0, I0, R0: initial conditions
        beta, gamma: model parameters
        N: total population
        n_days: total simulation duration in days
        step_sizes: list of dt values to test (e.g., [0.125, 0.25, 0.5, 1, 2, 4])

    Returns:
        results: dict with final I values and full trajectories per dt per method
    """
    results = {'euler': {}, 'rk4': {}}

    for dt in step_sizes:
        n_steps = int(n_days / dt)

        # Run both methods
        _, I_euler, _ = euler_sir(S0, I0, R0, beta, gamma, N, dt, n_steps)
        _, I_rk4,   _ = rk4_sir( S0, I0, R0, beta, gamma, N, dt, n_steps)

        results['euler'][dt] = I_euler
        results['rk4'][dt]   = I_rk4

    return results

In [ ]:
def plot_step_size_sensitivity(sensitivity_results, step_sizes, n_days,
                                save_path=None):
    """
    Visualize how step size affects the two methods.

    Produces a 2-panel figure:
      Left  — Final infected count vs. step size for each method
      Right — Full I(t) trajectories for each step size (Euler vs RK4)
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Panel 1: Final I value vs dt ──────────────────────────────────────
    ax = axes[0]
    euler_finals = [sensitivity_results['euler'][dt][-1] / 1e6 for dt in step_sizes]
    rk4_finals   = [sensitivity_results['rk4'][dt][-1]   / 1e6 for dt in step_sizes]

    ax.plot(step_sizes, euler_finals, 'bo-', label='Euler', linewidth=2, markersize=7)
    ax.plot(step_sizes, rk4_finals,   'rs-', label='RK4',   linewidth=2, markersize=7)
    ax.set_xlabel('Step Size dt (days)', fontsize=11)
    ax.set_ylabel('Final Infected I (millions)', fontsize=11)
    ax.set_title('Step-Size Sensitivity: Final Value', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_xscale('log')

    # Annotate instability region
    for i, dt in enumerate(step_sizes):
        if np.isnan(euler_finals[i]) or euler_finals[i] < 0:
            ax.annotate('UNSTABLE', (dt, 0), color='red', fontsize=9,
                        ha='center', va='bottom')

    # ── Panel 2: Full trajectories for each dt ────────────────────────────
    ax = axes[1]
    colors = plt.cm.Blues(np.linspace(0.4, 1.0, len(step_sizes)))

    for color, dt in zip(colors, step_sizes):
        I_euler = sensitivity_results['euler'][dt]
        t_euler = np.linspace(0, n_days, len(I_euler))
        ax.plot(t_euler, I_euler / 1e6, '--', color=color, alpha=0.7,
                label=f'Euler dt={dt}')

    colors_r = plt.cm.Reds(np.linspace(0.4, 1.0, len(step_sizes)))
    for color, dt in zip(colors_r, step_sizes):
        I_rk4 = sensitivity_results['rk4'][dt]
        t_rk4 = np.linspace(0, n_days, len(I_rk4))
        ax.plot(t_rk4, I_rk4 / 1e6, '-', color=color, alpha=0.7,
                label=f'RK4 dt={dt}')

    ax.set_xlabel('Time (days)', fontsize=11)
    ax.set_ylabel('Infected I (millions)', fontsize=11)
    ax.set_title('Full Trajectories Across Step Sizes', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, ncol=2, loc='upper left')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    plt.show()
    return fig

In [ ]:
# ============================================================
# CONVERGENCE ANALYSIS
# ============================================================

def convergence_analysis(S0, I0, R0, beta, gamma, N, n_days, step_sizes):
    """
    Measure how fast each method converges to the 'true' solution
    as step size decreases.

    Strategy: use the finest step size (smallest dt) as the reference
    'true' solution, then measure error of coarser dt against it.

    This shows that RK4 converges at order 4 (error ∝ dt^4) while
    Euler converges at order 1 (error ∝ dt^1).

    Args:
        step_sizes: list of dt values, MUST be sorted smallest to largest

    Returns:
        dict with MAE per method per dt (relative to finest solution)
    """
    step_sizes_sorted = sorted(step_sizes)   # finest first
    dt_ref = step_sizes_sorted[0]            # reference (finest)
    n_ref  = int(n_days / dt_ref)

    # Reference solutions (finest dt = most accurate)
    _, I_euler_ref, _ = euler_sir(S0, I0, R0, beta, gamma, N, dt_ref, n_ref)
    _, I_rk4_ref,   _ = rk4_sir( S0, I0, R0, beta, gamma, N, dt_ref, n_ref)

    convergence = {'euler': {}, 'rk4': {}}

    for dt in step_sizes_sorted[1:]:   # skip the reference itself
        n_steps = int(n_days / dt)
        factor  = int(dt / dt_ref)     # how many fine steps per coarse step

        # Run coarse solutions
        _, I_euler_c, _ = euler_sir(S0, I0, R0, beta, gamma, N, dt, n_steps)
        _, I_rk4_c,   _ = rk4_sir( S0, I0, R0, beta, gamma, N, dt, n_steps)

        # Subsample reference to match coarse grid
        ref_euler_sub = I_euler_ref[::factor][:len(I_euler_c)]
        ref_rk4_sub   = I_rk4_ref[::factor][:len(I_rk4_c)]

        # MAE of coarse vs fine reference
        convergence['euler'][dt] = np.mean(np.abs(I_euler_c - ref_euler_sub))
        convergence['rk4'][dt]   = np.mean(np.abs(I_rk4_c   - ref_rk4_sub))

    return convergence

In [ ]:
def plot_convergence(convergence_results, save_path=None):
    """
    Log-log convergence plot: error vs step size.

    On a log-log plot:
      - Euler should give a slope of ~1  (1st order)
      - RK4 should give a slope of ~4   (4th order)
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    dts_e   = sorted(convergence_results['euler'].keys())
    dts_r   = sorted(convergence_results['rk4'].keys())
    errs_e  = [convergence_results['euler'][dt] for dt in dts_e]
    errs_r  = [convergence_results['rk4'][dt]   for dt in dts_r]

    ax.loglog(dts_e, errs_e, 'bo-', label='Euler',  linewidth=2, markersize=8)
    ax.loglog(dts_r, errs_r, 'rs-', label='RK4',    linewidth=2, markersize=8)

    # Reference slope lines
    dt_arr = np.array(dts_e, dtype=float)
    ax.loglog(dt_arr, errs_e[0] * (dt_arr / dts_e[0])**1,
              'b--', alpha=0.4, label='O(dt¹) reference')
    ax.loglog(dt_arr, errs_r[0] * (dt_arr / dts_r[0])**4,
              'r--', alpha=0.4, label='O(dt⁴) reference')

    # Compute and annotate empirical slopes
    if len(dts_e) >= 2:
        slope_e = np.polyfit(np.log(dts_e), np.log(errs_e), 1)[0]
        slope_r = np.polyfit(np.log(dts_r), np.log(errs_r), 1)[0]
        ax.text(0.05, 0.85, f'Euler slope ≈ {slope_e:.2f}',
                transform=ax.transAxes, color='blue', fontsize=11)
        ax.text(0.05, 0.78, f'RK4 slope   ≈ {slope_r:.2f}',
                transform=ax.transAxes, color='red',  fontsize=11)

    ax.set_xlabel('Step Size dt (days)', fontsize=11)
    ax.set_ylabel('MAE vs. Reference Solution', fontsize=11)
    ax.set_title('Convergence Analysis: Euler vs. RK4', fontsize=12, fontweight='bold')
    ax.legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    plt.show()
    return fig

In [ ]:
# ============================================================
# STABILITY ANALYSIS
# ============================================================

def stability_analysis(S0, I0, R0, beta, gamma, N, n_days, step_sizes):
    """
    Test whether each method remains stable (no blow-up, no negative values)
    across a range of step sizes.

    Euler is conditionally stable — at large dt it can produce negative
    populations or explosive growth. RK4 is more stable but can also
    fail at very large dt.

    Args:
        step_sizes: list of dt values to test

    Returns:
        stability_report: dict with stability status and diagnostics per method per dt
    """
    report = {'euler': {}, 'rk4': {}}

    for dt in step_sizes:
        n_steps = int(n_days / dt)

        for method_name, solver in [('euler', euler_sir), ('rk4', rk4_sir)]:
            S, I, R = solver(S0, I0, R0, beta, gamma, N, dt, n_steps)

            has_negative   = bool(np.any(I < 0) or np.any(S < 0))
            has_nan        = bool(np.any(np.isnan(I)))
            has_explosion  = bool(np.any(np.abs(I) > N * 10))   # 10x population
            final_val      = float(I[-1])

            stable = not (has_negative or has_nan or has_explosion)

            report[method_name][dt] = {
                'stable':        stable,
                'has_negative':  has_negative,
                'has_nan':       has_nan,
                'has_explosion': has_explosion,
                'final_I':       final_val,
                'min_I':         float(np.min(I)),
                'max_I':         float(np.max(I))
            }

    return report

In [ ]:
def plot_stability(stability_report, step_sizes, N=N_PHILIPPINES, save_path=None):
    """
    Visualize stability results as a color-coded table and final-value plot.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    methods     = ['euler', 'rk4']
    method_lbls = ['Euler', 'RK4']
    colors_map  = {True: 'green', False: 'red'}

    # ── Panel 1: color-coded stability table ──────────────────────────────
    ax = axes[0]
    ax.set_xlim(-0.5, len(step_sizes) - 0.5)
    ax.set_ylim(-0.5, len(methods) - 0.5)
    ax.set_xticks(range(len(step_sizes)))
    ax.set_xticklabels([f'dt={dt}' for dt in step_sizes], rotation=45, ha='right')
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(method_lbls)
    ax.set_title('Stability Map (Green = Stable, Red = Unstable)',
                 fontsize=12, fontweight='bold')

    for i, method in enumerate(methods):
        for j, dt in enumerate(step_sizes):
            stable = stability_report[method][dt]['stable']
            color  = colors_map[stable]
            rect   = plt.Rectangle((j - 0.4, i - 0.4), 0.8, 0.8,
                                    color=color, alpha=0.7)
            ax.add_patch(rect)
            label = 'OK' if stable else 'FAIL'
            ax.text(j, i, label, ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')

    # ── Panel 2: final I value vs dt (detect blow-up visually) ───────────
    ax = axes[1]
    for method, label, color, marker in [('euler', 'Euler', 'blue', 'o'),
                                          ('rk4',   'RK4',   'red',  's')]:
        finals = []
        for dt in step_sizes:
            val = stability_report[method][dt]['final_I']
            finals.append(val if not np.isnan(val) and abs(val) < N * 10 else np.nan)

        ax.plot(step_sizes, [v / 1e6 if v == v else np.nan for v in finals],
                f'{color[0]}{marker}-', label=label, linewidth=2, markersize=7)

    ax.set_xlabel('Step Size dt (days)', fontsize=11)
    ax.set_ylabel('Final Infected I (millions)', fontsize=11)
    ax.set_title('Final I Value vs Step Size\n(Divergence = instability)',
                 fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_xscale('log')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    plt.show()
    return fig

In [ ]:
def print_stability_report(stability_report, step_sizes):
    """Print a readable stability summary table to the console."""
    print("\n  {'Method':<8} {'dt':<8} {'Stable':<10} {'Min I':<20} {'Final I'}")
    print("  " + "-" * 65)
    for method in ['euler', 'rk4']:
        for dt in step_sizes:
            r = stability_report[method][dt]
            status   = "✓ OK"   if r['stable'] else "✗ FAIL"
            min_val  = f"{r['min_I']:,.0f}"
            final    = f"{r['final_I']:,.0f}" if not np.isnan(r['final_I']) else "NaN"
            print(f"  {method.upper():<8} {dt:<8} {status:<10} {min_val:<20} {final}")

In [ ]:
# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_trajectories(dates, S_euler, I_euler, R_euler,
                      S_rk4, I_rk4, R_rk4, I_observed,
                      save_path=None):
    """
    Create comprehensive trajectory comparison plot.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Panel 1: Full trajectory comparison (I compartment)
    ax = axes[0, 0]
    ax.plot(dates, I_observed, 'k-', linewidth=2, label='Observed', alpha=0.8)
    ax.plot(dates, I_euler, 'b--', linewidth=1.5, label='Euler', alpha=0.7)
    ax.plot(dates, I_rk4, 'r-.', linewidth=1.5, label='RK4', alpha=0.7)
    ax.set_ylabel('Cumulative Cases', fontsize=11)
    ax.set_title('SIR Model Fit: Infected Compartment', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left')
    ax.set_yscale('log')

    # Panel 2: All compartments (RK4)
    ax = axes[0, 1]
    ax.plot(dates, S_rk4/1e6, 'g-', label='Susceptible', alpha=0.7)
    ax.plot(dates, I_rk4/1e6, 'r-', label='Infected', alpha=0.7)
    ax.plot(dates, R_rk4/1e6, 'b-', label='Recovered', alpha=0.7)
    ax.set_ylabel('Population (millions)', fontsize=11)
    ax.set_title('SIR Compartments (RK4)', fontsize=12, fontweight='bold')
    ax.legend(loc='center right')

    # Panel 3: Error time series
    ax = axes[1, 0]
    error_euler = I_euler - I_observed
    error_rk4 = I_rk4 - I_observed
    ax.plot(dates, error_euler/1e3, 'b-', alpha=0.6, label='Euler error')
    ax.plot(dates, error_rk4/1e3, 'r-', alpha=0.6, label='RK4 error')
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.set_ylabel('Error (thousands)', fontsize=11)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_title('Prediction Error Over Time', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left')

    # Panel 4: Method-to-method difference
    ax = axes[1, 1]
    method_diff = I_euler - I_rk4
    ax.plot(dates, method_diff/1e3, 'purple', alpha=0.7)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.set_ylabel('Euler − RK4 (thousands)', fontsize=11)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_title('Numerical Method Divergence', fontsize=12, fontweight='bold')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to {save_path}")

    plt.show()

    return fig

In [ ]:
def plot_error_analysis(comparison, save_path=None):
    """
    Visualize error distribution and comparative metrics.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Error distribution histograms
    ax = axes[0]
    euler_errors = comparison['Euler_vs_Data']['Error_Array']
    rk4_errors = comparison['RK4_vs_Data']['Error_Array']

    ax.hist(euler_errors/1e3, bins=50, alpha=0.5, label='Euler', color='blue', density=True)
    ax.hist(rk4_errors/1e3, bins=50, alpha=0.5, label='RK4', color='red', density=True)
    ax.set_xlabel('Error (thousands)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title('Error Distribution', fontsize=12, fontweight='bold')
    ax.legend()

    # Absolute error over time
    ax = axes[1]
    ax.plot(np.abs(euler_errors)/1e3, alpha=0.6, label='Euler', color='blue')
    ax.plot(np.abs(rk4_errors)/1e3, alpha=0.6, label='RK4', color='red')
    ax.set_xlabel('Time (days)', fontsize=11)
    ax.set_ylabel('|Error| (thousands)', fontsize=11)
    ax.set_title('Absolute Error Trajectory', fontsize=12, fontweight='bold')
    ax.set_yscale('log')
    ax.legend()

    # Metric comparison bar chart
    ax = axes[2]
    metrics = ['MAE', 'RMSE', 'Max_Error']
    euler_vals = [comparison['Euler_vs_Data'][m]/1e3 for m in metrics]
    rk4_vals = [comparison['RK4_vs_Data'][m]/1e3 for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35

    ax.bar(x - width/2, euler_vals, width, label='Euler', color='blue', alpha=0.7)
    ax.bar(x + width/2, rk4_vals, width, label='RK4', color='red', alpha=0.7)

    ax.set_ylabel('Error (thousands)', fontsize=11)
    ax.set_title('Error Metrics Comparison', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()

    # Add value labels
    for i, (e, r) in enumerate(zip(euler_vals, rk4_vals)):
        ax.text(i - width/2, e + 5, f'{e:.0f}', ha='center', va='bottom', fontsize=9)
        ax.text(i + width/2, r + 5, f'{r:.0f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

    return fig

In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    """
    Execute complete analysis pipeline.
    """
    print("=" * 70)
    print("SIR MODEL NUMERICAL ANALYSIS: PHILIPPINE COVID-19 DATA")
    print("=" * 70)

    # ============================================================
    # 1. LOAD AND PREPROCESS DATA
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 1: Data Loading")
    print("=" * 70)

    dates, cases = load_covid_data()
    n_days = len(cases) - 1  # Number of integration steps
    print(f"Integration steps: {n_days}")

    # Initial conditions
    S0, I0, R0 = prepare_initial_conditions(cases[0])
    print(f"Initial conditions: S0={S0:.0f}, I0={I0:.0f}, R0={R0:.0f}")

    # ============================================================
    # 2. PARAMETER ESTIMATION (using RK4 for primary estimates)
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 2: Parameter Estimation")
    print("=" * 70)

    beta_hat, gamma_hat, R0_hat, opt_result = estimate_parameters(
        cases, S0, I0, R0, N_PHILIPPINES, DT, n_days,
        method='rk4', n_restarts=5
    )

    # Verify with Euler (should be very similar)
    print("\nVerifying with Euler method...")
    beta_e, gamma_e, R0_e, _ = estimate_parameters(
        cases, S0, I0, R0, N_PHILIPPINES, DT, n_days,
        method='euler', n_restarts=3
    )

    # Use RK4 estimates as consensus (more stable optimization)
    print(f"\nConsensus parameters: β={beta_hat:.4f}, γ={gamma_hat:.4f}, R₀={R0_hat:.3f}")

    # ============================================================
    # 3. FORWARD INTEGRATION WITH BOTH METHODS
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 3: Numerical Integration")
    print("=" * 70)

    # Euler integration
    print("Running Euler method...")
    S_euler, I_euler, R_euler = euler_sir(
        S0, I0, R0, beta_hat, gamma_hat, N_PHILIPPINES, DT, n_days
    )

    # RK4 integration
    print("Running RK4 method...")
    S_rk4, I_rk4, R_rk4 = rk4_sir(
        S0, I0, R0, beta_hat, gamma_hat, N_PHILIPPINES, DT, n_days
    )

    print(f"Euler final I: {I_euler[-1]:,.0f}")
    print(f"RK4 final I: {I_rk4[-1]:,.0f}")
    print(f"Observed final: {cases[-1]:,.0f}")

    # ============================================================
    # 4. ERROR ANALYSIS
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 4: Error Analysis")
    print("=" * 70)

    comparison = compare_methods(I_euler, I_rk4, cases)

    print("\nMethod-to-method comparison:")
    print(f"  Max |Euler - RK4|: {comparison['Max_Method_Diff']:,.0f} cases")
    print(f"  Mean |Euler - RK4|: {comparison['Mean_Method_Diff']:,.0f} cases")
    print(f"  Correlation: {comparison['Method_Correlation']:.6f}")

    print("\nEuler vs. observed:")
    for key in ['MAE', 'RMSE', 'Max_Error', 'MAPE']:
        val = comparison['Euler_vs_Data'][key]
        if key == 'MAPE':
            print(f"  {key}: {val:.2f}%")
        else:
            print(f"  {key}: {val:,.0f} cases")

    print("\nRK4 vs. observed:")
    for key in ['MAE', 'RMSE', 'Max_Error', 'MAPE']:
        val = comparison['RK4_vs_Data'][key]
        if key == 'MAPE':
            print(f"  {key}: {val:.2f}%")
        else:
            print(f"  {key}: {val:,.0f} cases")

    # ============================================================
    # 5. VISUALIZATION (original)
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 5: Visualization")
    print("=" * 70)

    # Convert dates for plotting
    plot_dates = pd.to_datetime(dates)

    # Main trajectory plot
    fig1 = plot_trajectories(
        plot_dates,
        S_euler, I_euler, R_euler,
        S_rk4, I_rk4, R_rk4,
        cases,
        save_path='sir_trajectories.png'
    )

    # Error analysis plot
    fig2 = plot_error_analysis(comparison, save_path='error_analysis.png')

    # ============================================================
    # 6.  STEP-SIZE SENSITIVITY ANALYSIS
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 6: Step-Size Sensitivity Analysis")
    print("=" * 70)

    # Test a range of step sizes from very fine to very coarse
    step_sizes = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]

    print(f"Testing step sizes: {step_sizes} days")
    sensitivity_results = step_size_sensitivity(
        S0, I0, R0, beta_hat, gamma_hat, N_PHILIPPINES, n_days, step_sizes
    )

    print("\nFinal Infected I per step size:")
    print(f"  {'dt':<8} {'Euler Final I':<20} {'RK4 Final I'}")
    print("  " + "-" * 48)
    for dt in step_sizes:
        e_final = sensitivity_results['euler'][dt][-1]
        r_final = sensitivity_results['rk4'][dt][-1]
        print(f"  {dt:<8} {e_final:<20,.0f} {r_final:,.0f}")

    fig3 = plot_step_size_sensitivity(
        sensitivity_results, step_sizes, n_days,
        save_path='step_size_sensitivity.png'
    )

    # ============================================================
    # 7.  CONVERGENCE ANALYSIS
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 7: Convergence Analysis")
    print("=" * 70)

    # Use fine-to-coarse range; finest dt = reference solution
    conv_step_sizes = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0]

    print("Computing convergence rates (reference = dt=0.125)...")
    convergence_results = convergence_analysis(
        S0, I0, R0, beta_hat, gamma_hat, N_PHILIPPINES, n_days, conv_step_sizes
    )

    print("\nConvergence MAE vs. reference solution:")
    print(f"  {'dt':<8} {'Euler MAE':<20} {'RK4 MAE'}")
    print("  " + "-" * 48)
    for dt in sorted(convergence_results['euler'].keys()):
        e_mae = convergence_results['euler'][dt]
        r_mae = convergence_results['rk4'][dt]
        print(f"  {dt:<8} {e_mae:<20,.2f} {r_mae:,.2f}")

    fig4 = plot_convergence(convergence_results, save_path='convergence_analysis.png')

    # ============================================================
    # 8.  STABILITY ANALYSIS
    # ============================================================
    print("\n" + "=" * 70)
    print("STEP 8: Stability Analysis")
    print("=" * 70)

    # Include large step sizes to provoke instability in Euler
    stab_step_sizes = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

    print(f"Testing stability across step sizes: {stab_step_sizes} days")
    stability_report = stability_analysis(
        S0, I0, R0, beta_hat, gamma_hat, N_PHILIPPINES, n_days, stab_step_sizes
    )

    print_stability_report(stability_report, stab_step_sizes)

    fig5 = plot_stability(stability_report, stab_step_sizes,
                          save_path='stability_analysis.png')

    # ============================================================
    # 9. SUMMARY AND CONCLUSIONS
    # ============================================================
    print("\n" + "=" * 70)
    print("SUMMARY AND CONCLUSIONS")
    print("=" * 70)

    print(f"""
    Parameter Estimates:
      - Transmission rate (β): {beta_hat:.4f} day⁻¹
      - Recovery rate (γ): {gamma_hat:.4f} day⁻¹
      - Basic reproduction number (R₀): {R0_hat:.3f}
      - Implied infectious period: {1/gamma_hat:.1f} days

    Numerical Method Comparison:
      - Euler MAE: {comparison['Euler_vs_Data']['MAE']:,.0f} cases
      - RK4 MAE: {comparison['RK4_vs_Data']['MAE']:,.0f} cases
      - Difference: {abs(comparison['RK4_vs_Data']['MAE'] - comparison['Euler_vs_Data']['MAE']):,.0f} cases ({abs(comparison['RK4_vs_Data']['MAE'] - comparison['Euler_vs_Data']['MAE'])/comparison['Euler_vs_Data']['MAE']*100:.2f}%)
      - Method correlation: {comparison['Method_Correlation']:.6f}

    Step-Size Sensitivity:
      - Both methods are stable at dt ≤ 1.0 day (standard choice)
      - Euler begins to deviate at larger step sizes
      - RK4 maintains accuracy across a wider range of dt

    Convergence Behavior:
      - Euler converges at ~O(dt¹): halving dt halves the error
      - RK4 converges at ~O(dt⁴): halving dt reduces error by ~16x
      - RK4 is dramatically more accurate per unit of computation

    Stability:
      - Euler is conditionally stable: fails at large dt
      - RK4 remains stable over a broader range of step sizes

    Key Finding:
      RK4's theoretical superiority is MASKED by dominant MODEL STRUCTURAL ERROR.
      Both methods exhibit ~72% mean percentage error vs. observed data.
      However, convergence and stability analyses clearly confirm RK4's
      mathematical advantage when the model structure is held constant.
      Investment priority: MODEL ENRICHMENT over numerical method refinement.
    """)

    print("=" * 70)
    print("Analysis complete.")
    print("=" * 70)

    return {
        'parameters': (beta_hat, gamma_hat, R0_hat),
        'trajectories': {
            'euler': (S_euler, I_euler, R_euler),
            'rk4':   (S_rk4,   I_rk4,   R_rk4)
        },
        'comparison':    comparison,
        'sensitivity':   sensitivity_results,
        'convergence':   convergence_results,
        'stability':     stability_report,
        'figures':       (fig1, fig2, fig3, fig4, fig5)
    }

In [ ]:
# Execute if run as script
if __name__ == '__main__':
    results = main()